# Phase 8b — Promotion TWFE Robustness Audit

Notebook นี้ตรวจข้อกังวลเรื่อง treatment-effect heterogeneity ของค่า **+28.4%** จาก Phase 8 สำหรับ `promotion_flag` ซึ่งเป็น treatment แบบเปิด/ปิดซ้ำ ไม่ใช่ staggered adoption ทางเดียว

ผลสรุป: หลังแก้ชนิดข้อมูลของ ID ให้เป็นตัวเลขตามข้อกำหนดจริงของ package แล้ว ทั้ง negative-weights diagnostic และ static DID_M/WAS alternative ทำงานได้ ข้อสรุปเดิมว่าเครื่องมือทั้งสามล้มเพราะสลับ treatment ถี่จึง **ไม่ถูกต้อง** — สาเหตุหลักของสองเครื่องมือแรกคือ string ID ถูก package แปลงเป็น missing ภายใน


In [1]:
import sys
sys.path.insert(0, '../src')

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

from causal.promotion_robustness import (
    build_group_panel, compute_twfe_weight_diagnostic,
    export_group_panel_csv, fit_parity_twfe, fit_promotion_distributed_lag,
)

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)

df = pd.read_csv('../data/processed/weekly_features.csv', parse_dates=['week'])
panel = build_group_panel(df)
print('panel shape:', panel.shape)
print('groups:', panel.group_id.nunique(), '| periods:', panel.time_id.nunique())
print('numeric IDs:', panel[['group_id', 'time_id', 'cluster_id']].dtypes.to_dict())


panel shape: (31027, 45)
groups: 270 | periods: 150
numeric IDs: {'group_id': dtype('int64'), 'time_id': dtype('int64'), 'cluster_id': dtype('int64')}


## 1. Parity TWFE

ใช้ fixed-effect unit ระดับ `sku × channel × region` (270 หน่วย) ซึ่งตรงกับระดับที่ treatment แปรผัน และ cluster standard errors ที่ SKU


In [2]:
twfe = fit_parity_twfe(panel)
coef = float(twfe.params['promotion_flag'])
lo, hi = twfe.conf_int().loc['promotion_flag']
print(f'TWFE log coefficient: {coef:.6f}')
print(f'Uplift: {100*np.expm1(coef):.2f}% [{100*np.expm1(lo):.2f}%, {100*np.expm1(hi):.2f}%]')


TWFE log coefficient: 0.250022
Uplift: 28.41% [27.55%, 29.26%]


## 2. Negative-weights diagnostic

คำนวณน้ำหนักของ treated cells โดย residualize `promotion_flag` ด้วย group FE, week FE และ `price_unit` แล้วหารด้วยผลรวม residualized treatment ใน treated cells ตาม Frisch–Waugh–Lovell decomposition การคำนวณนี้ตรวจ parity โดย reconstruct สัมประสิทธิ์ TWFE จาก outcome โดยตรง


In [3]:
weight_summary, weight_detail = compute_twfe_weight_diagnostic(panel)
display(pd.Series(weight_summary, name='value').to_frame())
assert np.isclose(weight_summary['reconstructed_twfe_coef'], coef)
assert np.isclose(weight_summary['sum_weights'], 1.0)


,value
n_obs,31027.000000
n_treated_cells,19032.000000
n_positive_weights,19032.000000
n_negative_weights,0.000000
negative_weight_pct,0.000000
sum_positive_weights,1.000000
sum_negative_weights,0.000000
sum_weights,1.000000
min_weight,0.000020
max_weight,0.000091


### Independent R validation

`analysis/r/promotion_twfe_weights.R` รัน official `TwoWayFEWeights` ผ่าน Docker บน panel เดียวกัน จาก repo root ใช้ `powershell -ExecutionPolicy Bypass -File analysis/r/run_twfe_weights_docker.ps1` บน Windows หรือ `bash analysis/r/run_twfe_weights_docker.sh` บน Bash; runner จะสร้างและลบ panel export/temporary Docker volume ให้อัตโนมัติ ผลที่บันทึกใน `reports/promotion_twfe_weights_result.json` ตรงกับ Python decomposition: treated 19,032 cells ได้ positive weight ทั้งหมด, negative weight = 0, ผลรวมน้ำหนัก = 1, beta = 0.250022 และ sensitivity threshold ที่ทำให้ ATT เฉลี่ยเป็นศูนย์ = 1.609335 log points


In [4]:
r_result_path = Path('../reports/promotion_twfe_weights_result.json')
if r_result_path.exists():
    display(pd.Series(json.loads(r_result_path.read_text()), name='R TwoWayFEWeights').to_frame())
else:
    print('Run analysis/r/run_twfe_weights_docker.sh to generate the independent R result.')


,R TwoWayFEWeights
beta,0.250022
n_obs,31027
n_groups,270
n_treated_cells,19032
n_positive_weights,19032
n_negative_weights,0
sum_positive_weights,1
sum_negative_weights,0
sensitivity_to_zero_att,1.609335
type,feTR


## 3. Static heterogeneity-robust alternative — DID_M / exact-match WAS

`Wald-TC` เหมาะกับ fuzzy/IV design แต่ `promotion_flag` เป็น observed binary treatment จึงใช้ DID_M-equivalent exact-match WAS จาก `did-multiplegt-stat` แทน รายงานทั้ง total specification (ไม่ควบคุมราคาที่อาจเป็น mediator), conditional specification และแยก switch-in/switch-out


In [5]:
from did_multiplegt_stat import DIDMultiplegtStat

configs = [
    ('all — total', None, None),
    ('all — conditional on price', ['price_unit'], None),
    ('switch-in only', None, 'up'),
    ('switch-out only', None, 'down'),
]
rows = []
for label, controls, switchers in configs:
    estimator = DIDMultiplegtStat(
        estimator=['was'], exact_match=True, cluster='cluster_id',
        controls=controls, switchers=switchers, placebo=0,
    )
    estimator.fit(panel, Y='log_units', ID='group_id', Time='time_id', D='promotion_flag')
    row = estimator.to_dataframe().loc['WAS']
    rows.append({
        'specification': label, 'log_coef': row['Estimate'], 'se': row['SE'],
        'ci_low': row['LB CI'], 'ci_high': row['UB CI'],
        'uplift_pct': 100*np.expm1(row['Estimate']),
        'switchers': int(row['Switchers']), 'stayers': int(row['Stayers']),
    })
was_results = pd.DataFrame(rows).set_index('specification')
display(was_results.round(4))


,log_coef,se,ci_low,ci_high,uplift_pct,switchers,stayers
specification,,,,,,,
all — total,0.2538,0.0034,0.2471,0.2606,28.8944,14527,16230
all — conditional on price,0.2550,0.0036,0.2480,0.2620,29.0486,14527,16230
switch-in only,0.2527,0.0047,0.2436,0.2618,28.7501,7251,4621
switch-out only,0.2553,0.0036,0.2482,0.2624,29.0811,7270,11608


## 4. Dynamic / pull-forward sensitivity — recurring-treatment distributed lag

เมื่อใช้ numeric IDs แล้ว `py-did-multiplegt-dyn==0.1.9` ยังชน package bug ที่ accumulator บวก `None` เมื่อ internal time bin ว่าง (`did_multiplegt_dyn_core.py`, การนับ placebo mask) แม้ `placebo=0` การทดลองแก้บรรทัดนั้นชั่วคราวทำให้ estimator รันได้ แต่ไม่เก็บผลดังกล่าว เพราะเป็นการแก้ third-party code ที่ยังไม่ได้ยืนยันกับ reference implementation และ event-time แบบ first switch ไม่ตอบ repeated-promotion pull-forward โดยตรง

เพื่อทดสอบคำถาม pull-forward โดยตรงขึ้น จึงใช้ supplementary distributed-lag TWFE ที่ใส่สถานะโปรโมชันปัจจุบัน, lag 1–4 สัปดาห์ และ placebo lead 1–2 สัปดาห์ พร้อม group/week fixed effects และ cluster SE ที่ SKU เลือก horizon 4 สัปดาห์เพราะ non-promotion run มี percentile 95 เท่ากับ 4 สัปดาห์; หลังจากนั้น episode ใหม่ซ้อนทับเกือบทั้งหมดจนแยก lag เดิมได้ยาก ผลนี้เป็น sensitivity analysis ภายใต้ strict/sequential exogeneity และ stable additive lag effects ไม่ใช่ตัวแทนของ dynamic dCDH ที่ผ่าน validation แล้ว


In [6]:
dynamic_rows = []
for label, controls in [('total', ()), ('conditional on price', ('price_unit',))]:
    model = fit_promotion_distributed_lag(panel, lags=4, leads=2, controls=controls)
    lag_terms = [f'promo_lag_{k}' for k in range(1, 5)]
    lag_sum = float(model.params[lag_terms].sum())
    ones = np.ones(len(lag_terms))
    lag_sum_se = float(np.sqrt(ones @ model.cov.loc[lag_terms, lag_terms].to_numpy() @ ones))
    dynamic_rows.append({
        'specification': label,
        'n_obs': int(model.nobs),
        'current_coef': model.params['promo_current'],
        'lag_1': model.params['promo_lag_1'],
        'lag_2': model.params['promo_lag_2'],
        'lag_3': model.params['promo_lag_3'],
        'lag_4': model.params['promo_lag_4'],
        'lag_sum_1_4': lag_sum,
        'lag_sum_se': lag_sum_se,
        'lag_sum_p': 2 * norm.sf(abs(lag_sum / lag_sum_se)),
        'lead_1': model.params['promo_lead_1'],
        'lead_1_se': model.std_errors['promo_lead_1'],
        'lead_1_p': model.pvalues['promo_lead_1'],
        'lead_2': model.params['promo_lead_2'],
        'lead_2_se': model.std_errors['promo_lead_2'],
        'lead_2_p': model.pvalues['promo_lead_2'],
        'joint_lags_p': float(model.wald_test(
            formula='promo_lag_1 = promo_lag_2 = promo_lag_3 = promo_lag_4 = 0'
        ).pval),
        'joint_leads_p': float(model.wald_test(
            formula='promo_lead_1 = promo_lead_2 = 0'
        ).pval),
    })
dynamic_results = pd.DataFrame(dynamic_rows).set_index('specification')
display(dynamic_results.round(4))


,n_obs,current_coef,lag_1,lag_2,lag_3,lag_4,lag_sum_1_4,lag_sum_se,lag_sum_p,lead_1,lead_1_se,lead_1_p,lead_2,lead_2_se,lead_2_p,joint_lags_p,joint_leads_p
specification,,,,,,,,,,,,,,,,,
total,29407,0.2492,-0.0019,-0.0029,-0.0004,-0.0022,-0.0074,0.0092,0.4205,-0.006,0.0035,0.0831,-0.0000,0.003,0.9897,0.8703,0.2224
conditional on price,29407,0.2492,-0.0019,-0.0029,-0.0004,-0.0023,-0.0075,0.0092,0.4182,-0.006,0.0035,0.0883,-0.0001,0.003,0.9862,0.8667,0.2336


ใน total specification ผลรวม lag 1–4 เท่ากับ **−0.00743 log points** (SE 0.00922, p = 0.421; 95% CI [−0.02551, 0.01065]) และ joint Wald test ของ lag ทั้งสี่ให้ p = 0.870 ส่วน placebo lead 1 เท่ากับ −0.00602 (SE 0.00348, p = 0.083), lead 2 เท่ากับ −0.00004 (SE 0.00300, p = 0.990) และ joint leads p = 0.222 การเพิ่ม current price control ให้ผลแทบไม่เปลี่ยน (lag sum −0.00745, p = 0.418)

จึง **ไม่พบหลักฐานเชิงสถิติของยอดขายถูกดึงมาจาก 1–4 สัปดาห์ถัดไปภายใต้แบบจำลองนี้** แต่ confidence interval และสมมติฐานของโมเดลหมายความว่านี่ไม่ใช่หลักฐานว่า pull-forward ไม่มีอยู่จริง และไม่ทำให้ contemporaneous uplift เป็น causal โดยอัตโนมัติ


## 5. Conclusion

- TWFE: 0.250022 log points = **+28.41%**
- exact-match WAS (total): 0.253823 log points = **+28.89%** [28.03%, 29.77%]
- switch-in และ switch-out ให้ผลใกล้กัน (0.2527 และ 0.2553 log points)
- negative treated-cell weights: **0 / 19,032** ทั้ง Python decomposition และ official R package
- distributed-lag sensitivity: lag sum 1–4 = **−0.00743** log points (p = 0.421); joint lags p = 0.870; placebo leads p = 0.222

ดังนั้นข้อกังวลเฉพาะเรื่อง negative-weight contamination ไม่ปรากฏใน panel นี้, static alternative ยืนยันขนาดผลใกล้ TWFE และ distributed-lag model ไม่พบหลักฐาน pull-forward ในช่วง 1–4 สัปดาห์ อย่างไรก็ตาม นี่ยังไม่ทำให้ผลเป็น causal โดยอัตโนมัติ: parallel trends/sequential exogeneity, timing ภายในสัปดาห์, synthetic-data design และความไม่แน่นอนของ dynamic specification ยังเป็นข้อจำกัดที่ต้องรายงาน
